In [4]:
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM read_csv_auto('data/olist_orders_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE items AS SELECT * FROM read_csv_auto('data/olist_order_items_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE payments AS SELECT * FROM read_csv_auto('data/olist_order_payments_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE products AS SELECT * FROM read_csv_auto('data/olist_products_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE categories AS SELECT * FROM read_csv_auto('data/product_category_name_translation.csv')")
print("Sab tables load ho gaye")

IOException: IO Error: No files found that match the pattern "data/olist_orders_dataset.csv"

LINE 1: CREATE OR REPLACE TABLE orders AS SELECT * FROM read_csv_auto('data/olist_orders_dataset.csv')
                                                        ^

In [5]:
import os
import duckdb
import pandas as pd

os.chdir(r"C:\Users\Lenovo\OneDrive\Desktop\margin-leakage-engine")
print(os.getcwd())

con = duckdb.connect("margin.duckdb")

C:\Users\Lenovo\OneDrive\Desktop\margin-leakage-engine


In [6]:
import duckdb
import pandas as pd

con = duckdb.connect("margin.duckdb")

In [7]:
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM read_csv_auto('data/olist_orders_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE items AS SELECT * FROM read_csv_auto('data/olist_order_items_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE payments AS SELECT * FROM read_csv_auto('data/olist_order_payments_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE products AS SELECT * FROM read_csv_auto('data/olist_products_dataset.csv')")
con.execute("CREATE OR REPLACE TABLE categories AS SELECT * FROM read_csv_auto('data/product_category_name_translation.csv')")
con.execute("CREATE OR REPLACE TABLE assumptions AS SELECT * FROM read_csv_auto('data/assumptions.csv')")
print("Sab tables load ho gaye")

Sab tables load ho gaye


In [8]:
for t in ["orders", "items", "payments", "products", "categories"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows")

orders: 99441 rows
items: 112650 rows
payments: 103886 rows
products: 32951 rows
categories: 71 rows


In [9]:
con.execute("""
    SELECT order_status, COUNT(*) AS n
    FROM orders
    GROUP BY 1
    ORDER BY 2 DESC
""").df()

,order_status,n
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [10]:
SELECT order_id, COUNT(*) FROM items GROUP BY 1 ORDER BY 2 DESC LIMIT 5


SyntaxError: Invalid star expression (588193304.py, line 1)

In [11]:
con.execute("CREATE OR REPLACE TABLE assumptions AS SELECT * FROM read_csv_auto('data/assumptions.csv')")

In [12]:
query = open("sql/01_order_margin.sql").read()
df = con.execute(query).df()

print("Total delivered item rows:", len(df))
print(df[["revenue", "cogs", "shipping", "return_allowance", "realized_margin"]].describe())
print("\nNegative margin items count:", (df["realized_margin"] < 0).sum())

Total delivered item rows: 110197
             revenue           cogs       shipping  return_allowance  \
count  110197.000000  110197.000000  110197.000000     110197.000000   
mean      119.980563      60.237256      19.948598          1.580809   
std       182.299446      91.661593      15.698136          2.459470   
min         0.850000       0.425000       0.000000          0.010625   
25%        39.900000      19.950000      13.080000          0.499600   
50%        74.900000      37.500000      16.260000          0.974250   
75%       134.170000      67.495000      21.150000          1.749875   
max      6735.000000    3367.500000     409.680000         84.187500   

       realized_margin  
count    110197.000000  
mean        -11.786100  
std          83.406015  
min        -263.847500  
25%         -46.218000  
50%         -32.423750  
75%          -6.617125  
max        3039.002500  

Negative margin items count: 86846


In [13]:
summary = (
    df.groupby("category_english")
      .agg(
          items=("realized_margin", "size"),
          total_revenue=("revenue", "sum"),
          total_margin=("realized_margin", "sum"),
          avg_margin=("realized_margin", "mean"),
      )
      .assign(margin_pct=lambda x: (x.total_margin / x.total_revenue) * 100)
      .sort_values("margin_pct")
)
summary.head(10)

,items,total_revenue,total_margin,avg_margin,margin_pct
category_english,,,,,
home_comfort_2,30,760.27,-1539.678375,-51.322612,-202.517313
flowers,33,1110.04,-1597.725500,-48.415924,-143.934047
diapers_and_hygiene,37,1500.79,-1663.764875,-44.966618,-110.859272
cds_dvds_musicals,14,730.00,-569.115000,-40.651071,-77.960959
christmas_supplies,150,8737.84,-6431.213000,-42.874753,-73.601863
food_drink,269,14942.88,-10560.236000,-39.257383,-70.670687
electronics,2729,155043.93,-106545.244125,-39.041863,-68.719391
food,499,28731.15,-18007.094375,-36.086361,-62.674464
drinks,361,21529.84,-12995.473000,-35.998540,-60.360286


In [14]:
# Create SQL views for both item-level and order-level margins
view_sql = open("sql/02_create_margin_views.sql").read()
con.execute(view_sql)
print("Views created successfully!")

Views created successfully!


In [16]:
monthly_margin = con.execute("""
    SELECT 
        STRFTIME(TRY_STRPTIME(order_purchase_timestamp, '%Y-%m-%d %H:%M:%S'), '%Y-%m') AS order_month,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(order_revenue), 2) AS total_revenue,
        ROUND(SUM(order_realized_margin), 2) AS total_margin,
        ROUND(SUM(order_realized_margin) / SUM(order_revenue) * 100, 2) AS margin_pct
    FROM v_order_margins
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()

# Remove initial setup months with very low volume
monthly_margin = monthly_margin[monthly_margin['order_month'] >= '2017-01']
monthly_margin.tail(12)

BinderException: Binder Error: No function matches the given name and argument types 'try_strptime(TIMESTAMP, STRING_LITERAL)'. You might need to add explicit type casts.
	Candidate functions:
	try_strptime(VARCHAR, VARCHAR) -> TIMESTAMP
	try_strptime(VARCHAR, VARCHAR[]) -> TIMESTAMP


LINE 3:         STRFTIME(TRY_STRPTIME(order_purchase_timestamp, '%Y-%m-%d %H:%M:...
                         ^

In [18]:
monthly_margin = con.execute("""
    SELECT 
        STRFTIME(order_purchase_timestamp, '%Y-%m') AS order_month,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(order_revenue), 2) AS total_revenue,
        ROUND(SUM(order_realized_margin), 2) AS total_margin,
        ROUND(SUM(order_realized_margin) / SUM(order_revenue) * 100, 2) AS margin_pct
    FROM v_order_margins
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()

# Remove initial setup months with very low volume
monthly_margin = monthly_margin[monthly_margin['order_month'] >= '2017-01']
monthly_margin.tail(12)

,order_month,total_orders,total_revenue,total_margin,margin_pct
11,2017-09,4150,607399.67,-35013.95,-5.76
12,2017-10,4478,648247.65,-50117.82,-7.73
13,2017-11,7289,987765.37,-111462.69,-11.28
14,2017-12,5513,726033.19,-73601.76,-10.14
15,2018-01,7069,924645.00,-109563.76,-11.85
16,2018-02,6555,826437.13,-119505.32,-14.46
17,2018-03,7003,953356.25,-108396.97,-11.37
18,2018-04,6798,973534.09,-77850.57,-8.00
19,2018-05,6749,977544.69,-66035.68,-6.76
20,2018-06,6099,856077.86,-88409.34,-10.33


In [19]:
worst_skus = con.execute("""
    SELECT 
        product_id,
        category_english,
        COUNT(*) AS units_sold,
        ROUND(AVG(revenue), 2) AS avg_price,
        ROUND(AVG(shipping), 2) AS avg_shipping,
        ROUND(SUM(revenue), 2) AS total_revenue,
        ROUND(SUM(realized_margin), 2) AS total_loss
    FROM v_order_item_margins
    GROUP BY product_id, category_english
    HAVING SUM(realized_margin) < 0
    ORDER BY total_loss ASC
    LIMIT 10
""").df()

worst_skus

,product_id,category_english,units_sold,avg_price,avg_shipping,total_revenue,total_loss
0,422879e10f46682990de24d770e7f83d,garden_tools,484,54.91,15.75,26577.22,-18867.65
1,389d119b48cf3043d311335e499d9c6b,garden_tools,390,54.71,18.48,21336.79,-16305.17
2,368c6c730842d78016ad823897a372db,garden_tools,388,54.27,17.85,21056.80,-16062.41
3,53759a2ecddad2bb87a079a1f1519f73,garden_tools,373,54.66,18.45,20387.20,-15592.27
4,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,71.35,13.64,37104.30,-14838.13
5,154e7e31ebfa092203795c972e5804a6,health_beauty,274,22.53,13.32,6173.26,-14048.25
6,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,332,137.41,40.17,45620.56,-12600.94
7,99a4788cb24856965c36a24e339b6058,bed_bath_table,477,88.15,16.47,42049.66,-11208.63
8,b532349fe46b38fbc7bb3914c1bdae07,furniture_decor,166,36.50,14.97,6059.12,-7803.56
9,7c1bd920dbdf22470b68bde975dd3ccf,health_beauty,220,59.94,16.13,13187.80,-7492.43


In [20]:
# Create Order Leakage Features Dataset
leakage_df = con.execute("""
    WITH payment_vouchers AS (
        SELECT 
            order_id,
            SUM(CASE WHEN payment_type = 'voucher' THEN payment_value ELSE 0 END) AS voucher_amount,
            SUM(payment_value) AS total_payment_value
        FROM payments
        GROUP BY order_id
    ),
    order_reviews AS (
        SELECT 
            order_id,
            MIN(review_score) AS min_review_score
        FROM read_csv_auto('data/olist_order_reviews_dataset.csv')
        GROUP BY order_id
    )
    SELECT 
        m.order_id,
        m.customer_id,
        m.order_purchase_timestamp,
        m.total_items,
        m.order_revenue,
        m.order_shipping,
        m.order_realized_margin,
        m.order_margin_pct,
        COALESCE(v.voucher_amount, 0) AS voucher_amount,
        ROUND(COALESCE(v.voucher_amount, 0) / NULLIF(m.order_revenue, 0) * 100, 2) AS voucher_pct,
        ROUND(m.order_shipping / NULLIF(m.order_revenue, 0) * 100, 2) AS shipping_ratio_pct,
        COALESCE(r.min_review_score, 5) AS review_score,
        CASE WHEN m.order_realized_margin < 0 THEN 1 ELSE 0 END AS is_negative_margin
    FROM v_order_margins m
    LEFT JOIN payment_vouchers v ON m.order_id = v.order_id
    LEFT JOIN order_reviews r ON m.order_id = r.order_id
""").df()

print("Leakage dataset shape:", leakage_df.shape)
leakage_df.head()

Leakage dataset shape: (96478, 13)


,order_id,customer_id,order_purchase_timestamp,total_items,order_revenue,order_shipping,order_realized_margin,order_margin_pct,voucher_amount,voucher_pct,shipping_ratio_pct,review_score,is_negative_margin
0,dbfc85111e9812da1a864f78aaf6d282,6f831ab314ae38169aef4169d06d2429,2017-11-22 11:56:01,1,35.0,7.78,-40.71750,-116.34,0.0,0.0,22.23,5,1
1,fd94e66307d89e3285cde20f9766bafd,ccd36c540240c744dd26146ce739e082,2018-08-25 14:20:27,1,389.5,13.75,144.63250,37.13,0.0,0.0,3.53,5,0
2,6c3416485c9a6bcd0118e636c4083591,70a9efc5396b189e1992166fbe612bde,2018-04-05 20:12:22,1,699.0,17.33,273.43250,39.12,0.0,0.0,2.48,5,0
3,94072af7dce5862d93084c2494df0a38,19d69d5009cad1024fa24c7acfde2268,2018-05-01 11:07:25,1,99.9,0.00,-1.29875,-1.30,0.0,0.0,0.00,5,1
4,f3e2f155986b15b3fc25a97303ab4440,01414201d70bd508e1eee33ff67d2453,2018-02-22 11:59:21,2,107.8,30.26,-77.70750,-72.08,0.0,0.0,28.07,4,1


In [21]:
import numpy as np
from sklearn.ensemble import IsolationForest

# Clean data for modeling
df_clean = leakage_df.dropna(subset=['order_revenue', 'order_realized_margin', 'shipping_ratio_pct']).copy()

# 1. Statistical Baseline: IQR Method on Realized Margin
Q1 = df_clean['order_realized_margin'].quantile(0.25)
Q3 = df_clean['order_realized_margin'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR

df_clean['iqr_anomaly'] = (df_clean['order_realized_margin'] < lower_bound).astype(int)

# 2. Machine Learning: Isolation Forest (Multi-dimensional Anomaly Detection)
features = ['order_revenue', 'order_shipping', 'order_realized_margin', 'shipping_ratio_pct', 'voucher_pct']
iso = IsolationForest(contamination=0.03, random_state=42)
df_clean['iso_forest_anomaly'] = iso.fit_predict(df_clean[features])
# Convert: -1 (anomaly) -> 1, 1 (normal) -> 0
df_clean['iso_forest_anomaly'] = (df_clean['iso_forest_anomaly'] == -1).astype(int)

# 3. Model Comparison Matrix
comparison = pd.crosstab(
    df_clean['iqr_anomaly'], 
    df_clean['iso_forest_anomaly'], 
    rownames=['IQR (Statistical)'], 
    colnames=['Isolation Forest (ML)']
)
print("=== Anomaly Detection Comparison ===")
print(comparison)

print(f"\nIQR flagged anomalies: {df_clean['iqr_anomaly'].sum()}")
print(f"Isolation Forest flagged anomalies: {df_clean['iso_forest_anomaly'].sum()}")
print(f"Both agreed (High-Confidence Leaks): {((df_clean['iqr_anomaly'] == 1) & (df_clean['iso_forest_anomaly'] == 1)).sum()}")

=== Anomaly Detection Comparison ===
Isolation Forest (ML)      0     1
IQR (Statistical)                 
0                      92005  2403
1                       1578   492

IQR flagged anomalies: 2070
Isolation Forest flagged anomalies: 2895
Both agreed (High-Confidence Leaks): 492


In [22]:
# Profile: Normal orders vs High-Confidence Leakages
df_clean['leakage_type'] = np.where(
    (df_clean['iqr_anomaly'] == 1) & (df_clean['iso_forest_anomaly'] == 1), 'High-Confidence Leak',
    np.where((df_clean['iqr_anomaly'] == 1) | (df_clean['iso_forest_anomaly'] == 1), 'Suspect Leak', 'Normal')
)

leakage_profile = df_clean.groupby('leakage_type').agg(
    orders=('order_id', 'count'),
    avg_revenue=('order_revenue', 'mean'),
    avg_shipping=('order_shipping', 'mean'),
    avg_shipping_pct=('shipping_ratio_pct', 'mean'),
    avg_voucher_pct=('voucher_pct', 'mean'),
    avg_margin=('order_realized_margin', 'mean'),
    total_margin_loss=('order_realized_margin', lambda x: x[x < 0].sum())
).round(2)

leakage_profile

,orders,avg_revenue,avg_shipping,avg_shipping_pct,avg_voucher_pct,avg_margin,total_margin_loss
leakage_type,,,,,,,
High-Confidence Leak,492,225.51,131.17,109.43,16.48,-227.16,-111761.98
Normal,92005,116.90,20.58,29.71,3.18,-18.30,-2780003.39
Suspect Leak,3981,591.70,60.46,47.24,24.84,124.67,-287117.62


In [ ]:
# Root-Cause by Product Category & Basket Size
con.register("df_leakage_tagged", df_clean)

root_cause_category = con.execute("""
    SELECT 
        m.category_english,
        COUNT(DISTINCT d.order_id) AS total_orders
        SUM(CASE WHEN d.leakage_type = 'High-Confidence Leak' THEN 1 ELSE 0 END) AS high_conf_leaks,
        ROUND(AVG(d.order_shipping), 2) AS avg_shipping_cost,
        ROUND(AVG(d.shipping_ratio_pct), 2) AS avg_shipping_pct_of_rev,
        ROUND(AVG(d.voucher_pct), 2) AS avg_voucher_pct,
        ROUND(AVG(d.order_realized_margin), 2) AS avg_order_margin,
        ROUND(SUM(d.order_realized_margin), 2) AS total_category_margin
    FROM df_leakage_tagged d
    JOIN v_order_item_margins m ON d.order_id = m.order_id
    GROUP BY m.category_english
    ORDER BY high_conf_leaks DESC
    LIMIT 10
""").df()

root_cause_category

,category_english,total_orders,high_conf_leaks,avg_shipping_cost,avg_shipping_pct_of_rev,avg_voucher_pct,avg_order_margin,total_category_margin
0,furniture_decor,6307,306.0,34.82,34.12,4.39,-56.62,-461987.64
1,housewares,5743,264.0,31.94,39.90,5.20,-50.76,-344895.18
2,bed_bath_table,9272,152.0,26.26,28.12,4.38,-36.81,-403136.75
3,garden_tools,3448,151.0,34.51,33.10,3.99,-44.08,-188117.64
4,computers_accessories,6530,144.0,28.16,29.40,3.24,-37.88,-289542.11
5,office_furniture,1254,138.0,79.80,27.34,2.14,-33.07,-55167.29
6,health_beauty,8647,109.0,23.10,29.74,2.91,-8.43,-79744.86
7,sports_leisure,7530,104.0,24.51,29.12,3.74,-23.75,-200258.73
8,telephony,4093,60.0,19.46,50.63,4.38,-40.78,-180643.85
9,auto,3810,58.0,27.83,32.96,3.88,-8.61,-35655.08


In [26]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Filter around Black Friday 2017 (Oct 2017 to Dec 2017)
did_df = con.execute("""
    SELECT 
        d.order_id,
        d.order_purchase_timestamp,
        m.category_english,
        d.order_revenue,
        d.order_realized_margin,
        d.voucher_pct,
        -- Post-Campaign period: After Nov 24, 2017 (Black Friday)
        CASE WHEN d.order_purchase_timestamp >= '2017-11-24' THEN 1 ELSE 0 END AS post_campaign,
        -- Treatment Group: furniture_decor (heavy promo target), Control Group: health_beauty
        CASE WHEN m.category_english = 'furniture_decor' THEN 1 ELSE 0 END AS treatment
    FROM df_leakage_tagged d
    JOIN v_order_item_margins m ON d.order_id = m.order_id
    WHERE d.order_purchase_timestamp BETWEEN '2017-10-01' AND '2017-12-31'
      AND m.category_english IN ('furniture_decor', 'health_beauty')
""").df()

# Econometric DiD Model: Margin ~ Post + Treatment + (Post * Treatment)
model = smf.ols("order_realized_margin ~ post_campaign + treatment + post_campaign:treatment", data=did_df).fit()

print("=== Difference-in-Differences (DiD) Econometric Model ===")
print(model.summary().tables[1])

ModuleNotFoundError: No module named 'statsmodels'

In [1]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Filter around Black Friday 2017 (Oct 2017 to Dec 2017)
did_df = con.execute("""
    SELECT 
        d.order_id,
        d.order_purchase_timestamp,
        m.category_english,
        d.order_revenue,
        d.order_realized_margin,
        d.voucher_pct,
        -- Post-Campaign period: After Nov 24, 2017 (Black Friday)
        CASE WHEN d.order_purchase_timestamp >= '2017-11-24' THEN 1 ELSE 0 END AS post_campaign,
        -- Treatment Group: furniture_decor (heavy promo target), Control Group: health_beauty
        CASE WHEN m.category_english = 'furniture_decor' THEN 1 ELSE 0 END AS treatment
    FROM df_leakage_tagged d
    JOIN v_order_item_margins m ON d.order_id = m.order_id
    WHERE d.order_purchase_timestamp BETWEEN '2017-10-01' AND '2017-12-31'
      AND m.category_english IN ('furniture_decor', 'health_beauty')
""").df()

# Econometric DiD Model: Margin ~ Post + Treatment + (Post * Treatment)
model = smf.ols("order_realized_margin ~ post_campaign + treatment + post_campaign:treatment", data=did_df).fit()

print("=== Difference-in-Differences (DiD) Econometric Model ===")
print(model.summary().tables[1])

NameError: name 'con' is not defined

In [3]:
import duckdb
import statsmodels.api as sm
import statsmodels.formula.api as smf

con = duckdb.connect("margin.duckdb")

# Self-contained DiD query directly from database views
did_df = con.execute("""
    WITH payment_vouchers AS (
        SELECT 
            order_id,
            SUM(CASE WHEN payment_type = 'voucher' THEN payment_value ELSE 0 END) AS voucher_amount
        FROM payments
        GROUP BY order_id
    )
    SELECT 
        m.order_id,
        m.order_purchase_timestamp,
        m.category_english,
        m.revenue AS item_revenue,
        m.realized_margin AS item_realized_margin,
        ROUND(COALESCE(v.voucher_amount, 0) / NULLIF(m.revenue, 0) * 100, 2) AS voucher_pct,
        -- Post-Campaign period: After Nov 24, 2017 (Black Friday)
        CASE WHEN m.order_purchase_timestamp >= '2017-11-24' THEN 1 ELSE 0 END AS post_campaign,
        -- Treatment Group: furniture_decor (heavy promo target), Control Group: health_beauty
        CASE WHEN m.category_english = 'furniture_decor' THEN 1 ELSE 0 END AS treatment
    FROM v_order_item_margins m
    LEFT JOIN payment_vouchers v ON m.order_id = v.order_id
    WHERE m.order_purchase_timestamp BETWEEN '2017-10-01' AND '2017-12-31'
      AND m.category_english IN ('furniture_decor', 'health_beauty')
""").df()

# Econometric DiD Model: Margin ~ Post + Treatment + (Post * Treatment)
model = smf.ols("item_realized_margin ~ post_campaign + treatment + post_campaign:treatment", data=did_df).fit()

print("=== Difference-in-Differences (DiD) Econometric Model ===")
print(model.summary().tables[1])

CatalogException: Catalog Error: Table with name v_order_item_margins does not exist!
Did you mean "pg_description"?

LINE 20:     FROM v_order_item_margins m
                  ^

In [4]:
import duckdb
import statsmodels.api as sm
import statsmodels.formula.api as smf

con = duckdb.connect("margin.duckdb")

# Self-contained DiD query directly from database views
did_df = con.execute("""
    WITH payment_vouchers AS (
        SELECT 
            order_id,
            SUM(CASE WHEN payment_type = 'voucher' THEN payment_value ELSE 0 END) AS voucher_amount
        FROM payments
        GROUP BY order_id
    )
    SELECT 
        m.order_id,
        m.order_purchase_timestamp,
        m.category_english,
        m.revenue AS item_revenue,
        m.realized_margin AS item_realized_margin,
        ROUND(COALESCE(v.voucher_amount, 0) / NULLIF(m.revenue, 0) * 100, 2) AS voucher_pct,
        -- Post-Campaign period: After Nov 24, 2017 (Black Friday)
        CASE WHEN m.order_purchase_timestamp >= '2017-11-24' THEN 1 ELSE 0 END AS post_campaign,
        -- Treatment Group: furniture_decor (heavy promo target), Control Group: health_beauty
        CASE WHEN m.category_english = 'furniture_decor' THEN 1 ELSE 0 END AS treatment
    FROM v_order_item_margins m
    LEFT JOIN payment_vouchers v ON m.order_id = v.order_id
    WHERE m.order_purchase_timestamp BETWEEN '2017-10-01' AND '2017-12-31'
      AND m.category_english IN ('furniture_decor', 'health_beauty')
""").df()

# Econometric DiD Model: Margin ~ Post + Treatment + (Post * Treatment)
model = smf.ols("item_realized_margin ~ post_campaign + treatment + post_campaign:treatment", data=did_df).fit()

print("=== Difference-in-Differences (DiD) Econometric Model ===")
print(model.summary().tables[1])

CatalogException: Catalog Error: Table with name v_order_item_margins does not exist!
Did you mean "pg_description"?

LINE 20:     FROM v_order_item_margins m
                  ^

In [5]:
import duckdb
import statsmodels.api as sm
import statsmodels.formula.api as smf

con = duckdb.connect("margin.duckdb")

# 100% standalone DiD query directly from base tables
did_df = con.execute("""
    WITH order_vouchers AS (
        SELECT order_id, SUM(CASE WHEN payment_type = 'voucher' THEN payment_value ELSE 0 END) AS voucher_val
        FROM payments
        GROUP BY order_id
    )
    SELECT 
        i.order_id,
        o.order_purchase_timestamp,
        c.product_category_name_english AS category,
        i.price AS revenue,
        -- Margin = Revenue - COGS(50%) - CAC(50) - Shipping - Return Allowance(5% * 25% * Rev)
        (i.price - (i.price * COALESCE(a.cogs_pct, 0.50)) - 50.0 - i.freight_value - (i.price * 0.05 * 0.25)) AS item_margin,
        -- Post Black Friday (Nov 24, 2017)
        CASE WHEN o.order_purchase_timestamp >= '2017-11-24' THEN 1 ELSE 0 END AS post_campaign,
        -- Treatment: furniture_decor vs Control: health_beauty
        CASE WHEN c.product_category_name_english = 'furniture_decor' THEN 1 ELSE 0 END AS treatment
    FROM items i
    JOIN orders o ON i.order_id = o.order_id
    JOIN products p ON i.product_id = p.product_id
    JOIN categories c ON p.product_category_name = c.product_category_name
    LEFT JOIN assumptions a ON c.product_category_name_english = a.category_english
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp BETWEEN '2017-10-01' AND '2017-12-31'
      AND c.product_category_name_english IN ('furniture_decor', 'health_beauty')
""").df()

# Econometric DiD Model: Margin ~ Post + Treatment + (Post * Treatment)
model = smf.ols("item_margin ~ post_campaign + treatment + post_campaign:treatment", data=did_df).fit()

print("=== Difference-in-Differences (DiD) Econometric Model ===")
print(model.summary().tables[1])

CatalogException: Catalog Error: Table with name items does not exist!
Did you mean "sqlite_master"?

LINE 18:     FROM items i
                  ^

In [6]:
import duckdb
import statsmodels.api as sm
import statsmodels.formula.api as smf

con = duckdb.connect()  # clean in-memory connection

# Direct CSV-based DiD query (No table dependency, guaranteed to run!)
did_df = con.execute("""
    WITH order_vouchers AS (
        SELECT order_id, SUM(CASE WHEN payment_type = 'voucher' THEN payment_value ELSE 0 END) AS voucher_val
        FROM read_csv_auto('data/olist_order_payments_dataset.csv')
        GROUP BY order_id
    )
    SELECT 
        i.order_id,
        o.order_purchase_timestamp,
        c.product_category_name_english AS category,
        i.price AS revenue,
        -- Margin = Revenue - COGS(50%) - CAC(50) - Shipping - Return Allowance(5% * 25% * Rev)
        (i.price - (i.price * COALESCE(a.cogs_pct, 0.50)) - 50.0 - i.freight_value - (i.price * 0.05 * 0.25)) AS item_margin,
        -- Post Black Friday (Nov 24, 2017)
        CASE WHEN o.order_purchase_timestamp >= '2017-11-24' THEN 1 ELSE 0 END AS post_campaign,
        -- Treatment: furniture_decor vs Control: health_beauty
        CASE WHEN c.product_category_name_english = 'furniture_decor' THEN 1 ELSE 0 END AS treatment
    FROM read_csv_auto('data/olist_order_items_dataset.csv') i
    JOIN read_csv_auto('data/olist_orders_dataset.csv') o ON i.order_id = o.order_id
    JOIN read_csv_auto('data/olist_products_dataset.csv') p ON i.product_id = p.product_id
    JOIN read_csv_auto('data/product_category_name_translation.csv') c ON p.product_category_name = c.product_category_name
    LEFT JOIN read_csv_auto('data/assumptions.csv') a ON c.product_category_name_english = a.category_english
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp BETWEEN '2017-10-01' AND '2017-12-31'
      AND c.product_category_name_english IN ('furniture_decor', 'health_beauty')
""").df()

# Econometric DiD Model: Margin ~ Post + Treatment + (Post * Treatment)
model = smf.ols("item_margin ~ post_campaign + treatment + post_campaign:treatment", data=did_df).fit()

print("=== Difference-in-Differences (DiD) Econometric Model ===")
print(model.summary().tables[1])

IOException: IO Error: No files found that match the pattern "data/olist_order_items_dataset.csv"

LINE 18:     FROM read_csv_auto('data/olist_order_items_dataset.csv') i
                  ^

In [ ]:
import duckdb
import statsmodels.api as sm
import statsmodels.formula.api as smf

con = duckdb.connect()

# Direct CSV-based DiD query with correct relative path
did_df = con.execute("""
    WITH order_vouchers AS (
        SELECT order_id, SUM(CASE WHEN payment_type = 'voucher' THEN payment_value ELSE 0 END) AS voucher_val
        FROM read_csv_auto('../data/olist_order_payments_dataset.csv')
        GROUP BY order_id
    )
    SELECT 
        i.order_id,
        o.order_purchase_timestamp,
        c.product_category_name_english AS category,
        i.price AS revenue,
        -- Margin = Revenue - COGS(50%) - CAC(50) - Shipping - Return Allowance(5% * 25% * Rev)
        (i.price - (i.price * COALESCE(a.cogs_pct, 0.50)) - 50.0 - i.freight_value - (i.price * 0.05 * 0.25)) AS item_margin,
        -- Post Black Friday (Nov 24, 2017)
        CASE WHEN o.order_purchase_timestamp >= '2017-11-24' THEN 1 ELSE 0 END AS post_campaign,
        -- Treatment: furniture_decor vs Control: health_beauty
        CASE WHEN c.product_category_name_english = 'furniture_decor' THEN 1 ELSE 0 END AS treatment
    FROM read_csv_auto('../data/olist_order_items_dataset.csv') i
    JOIN read_csv_auto('../data/olist_orders_dataset.csv') o ON i.order_id = o.order_id
    JOIN read_csv_auto('../data/olist_products_dataset.csv') p ON i.product_id = p.product_id
    JOIN read_csv_auto('../data/product_category_name_translation.csv') c ON p.product_category_name = c.product_category_name
    LEFT JOIN read_csv_auto('../data/assumptions.csv') a ON c.product_category_name_english = a.category_english
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp BETWEEN '2017-10-01' AND '2017-12-31'
      AND c.product_category_name_english IN ('furniture_decor', 'health_beauty')
""").df()

# Econometric DiD Model: Margin ~ Post + Treatment + (Post * Treatment)
model = smf.ols("item_margin ~ post_campaign + treatment + post_campaign:treatment", data=did_df).fit()

print("=== Difference-in-Differences (DiD) Econometric Model ===")
print(model.summary().tables[1])

=== Difference-in-Differences (DiD) Econometric Model ===
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   1.1004      2.943      0.374      0.708      -4.670       6.871
post_campaign               0.5028      4.052      0.124      0.901      -7.442       8.447
treatment                 -32.1732      3.993     -8.057      0.000     -40.003     -24.344
post_campaign:treatment     2.3029      5.594      0.412      0.681      -8.666      13.272


: 

In [ ]:
# Verification & Reconciliation: Baseline Totals & Critical Leaks
import duckdb
import pandas as pd

con = duckdb.connect()

summary_check = con.execute('''
    SELECT 
        i.order_id,
        i.price AS revenue,
        (i.price * COALESCE(a.cogs_pct, 0.50)) AS cogs,
        50.0 AS ad_cac,
        i.freight_value AS freight_cost,
        (i.price * 0.05 * 0.25) AS return_allowance,
        (i.price - (i.price * COALESCE(a.cogs_pct, 0.50)) - 50.0 - i.freight_value - (i.price * 0.05 * 0.25)) AS realized_margin,
        CASE WHEN (i.price - (i.price * COALESCE(a.cogs_pct, 0.50)) - 50.0 - i.freight_value - (i.price * 0.05 * 0.25)) < -100 THEN 1 ELSE 0 END AS is_critical_leak
    FROM read_csv_auto('../data/olist_order_items_dataset.csv') i
    JOIN read_csv_auto('../data/olist_orders_dataset.csv') o ON i.order_id = o.order_id
    JOIN read_csv_auto('../data/olist_products_dataset.csv') p ON i.product_id = p.product_id
    JOIN read_csv_auto('../data/product_category_name_translation.csv') c ON p.product_category_name = c.product_category_name
    LEFT JOIN read_csv_auto('../data/assumptions.csv') a ON c.product_category_name_english = a.category_english
    WHERE o.order_status = 'delivered'
''').df()

print('=== EMPIRICAL FINANCIAL TOTALS ===')
print('Total Delivered Items:', len(summary_check))
print('Total Orders:', summary_check['order_id'].nunique())
print('Total Revenue: R$', round(summary_check['revenue'].sum(), 2))
print('Total Realized Margin: R$', round(summary_check['realized_margin'].sum(), 2))
print('Net Margin %:', round(summary_check['realized_margin'].sum() / summary_check['revenue'].sum() * 100, 2), '%')
print('Critical Leaks (< -R$ 100):', summary_check['is_critical_leak'].sum())

# Bulky Category Freight Savings Base Calculation
bulky_cats = ['furniture_decor', 'office_furniture', 'housewares']
bulky_freight = con.execute('''
    SELECT SUM(i.freight_value) AS total_bulky_freight
    FROM read_csv_auto('../data/olist_order_items_dataset.csv') i
    JOIN read_csv_auto('../data/olist_orders_dataset.csv') o ON i.order_id = o.order_id
    JOIN read_csv_auto('../data/olist_products_dataset.csv') p ON i.product_id = p.product_id
    JOIN read_csv_auto('../data/product_category_name_translation.csv') c ON p.product_category_name = c.product_category_name
    WHERE o.order_status = 'delivered'
      AND c.product_category_name_english IN ('furniture_decor', 'office_furniture', 'housewares')
''').fetchone()[0]

print('\n=== BULKY FREIGHT OPTIMIZATION RECONCILIATION ===')
print('Total Bulky Freight Base: R$', round(bulky_freight, 2))
print('EBITDA Recovered at 10% Freight Cut: R$', round(bulky_freight * 0.10, 2))


=== EMPIRICAL FINANCIAL TOTALS ===
Total Delivered Items: 108638
Total Orders: 95127
Total Revenue: R$ 13045530.9
Total Realized Margin: R$ -1270308.57
Net Margin %: -9.74 %
Critical Leaks (< -R$ 100): 118

=== BULKY FREIGHT OPTIMIZATION RECONCILIATION ===
Total Bulky Freight Base: R$ 378222.84
EBITDA Recovered at 10% Freight Cut: R$ 37822.28
